# Part 3 — Work IQ MCP

**MCP (Model Context Protocol)** exposes Work IQ as a **tool** to any MCP-capable agent (GitHub
Copilot CLI, Foundry agents, Agent Framework, …). Use MCP when your agent treats Work IQ as an
**agent-to-tool** dependency.

The 10 generic verbs:

| Read / discover | Write |
|---|---|
| `ask`, `fetch`, `search_paths`, `get_schema`, `call_function`, `list_agents` | `create_entity`, `update_entity`, `delete_entity`, `do_action` |

Key idea: **the tool surface never grows** as new data sources are added — you discover new
*paths* at runtime with `get_schema`.

In [1]:
import sys, os, json
sys.path.append(os.path.abspath(".."))
from notebooks._shared import get_user_token, call_mcp, call_tool

token = get_user_token()
# Work IQ speaks MCP (JSON-RPC 2.0) at {gateway}/mcp and streams responses over SSE.
# call_mcp / call_tool in _shared.py handle the envelope + `data:` frame parsing.
def mcp_call(name, arguments):
    return call_tool(token, name, arguments)
print("Signed in - Work IQ MCP helper ready. Token:", token[:16], "...")

Signed in - Work IQ MCP helper ready. Token: eyJ0eXAiOiJKV1Qi ...


## 1. List the tools the server offers

In [2]:
tools = call_mcp(token, "tools/list", {})
for tdef in tools["result"]["tools"]:
    req = tdef.get("inputSchema", {}).get("required", [])
    print(f"{tdef['name']:<14} required={req}")

ask            required=['question']
list_agents    required=[]
delete_entity  required=['entityUrl']
do_action      required=['actionUrl', 'jsonBody']
create_entity  required=['parentUrl', 'jsonBody']
get_schema     required=[]
search_paths   required=[]
call_function  required=['functionUrl']
fetch          required=['entityUrls']
update_entity  required=['entityUrl', 'jsonBody']


## 2. Runtime schema discovery with `get_schema`

Discover the shape of a path before you call it — no pre-baked models.

In [3]:
# get_schema needs operationType: one of fetch | create | update.
schema = mcp_call("get_schema", {"path": "/me/messages", "operationType": "fetch"})
print(schema["result"]["content"][0]["text"][:1200])

; CDDL definition for microsoft.graph.messageCollectionResponse

microsoft-graph-messageCollectionResponse = {
  BaseCollectionPaginationCountResponse,
  ? "value": [* microsoft-graph-message]
}

BaseCollectionPaginationCountResponse = {
  ? "@odata.count": (int / null),
  ? "@odata.nextLink": (tstr / null)
}

microsoft-graph-message = {
  microsoft-graph-outlookItem,
  ? "bccRecipients": [* microsoft-graph-recipient],
  ? "body": microsoft-graph-itemBody / { * tstr => any },
  ? "bodyPreview": (tstr / null),
  ? "ccRecipients": [* microsoft-graph-recipient],
  ? "conversationId": (tstr / null),
  ? "conversationIndex": (bstr / null),
  ? "flag": microsoft-graph-followupFlag / { * tstr => any },
  ? "from": microsoft-graph-recipient / { * tstr => any },
  ? "hasAttachments": (bool / null),
  ? "importance": microsoft-graph-importance / { * tstr => any },
  ? "inferenceClassification": microsoft-graph-inferenceClassificationType / { * tstr => any },
  ? "internetMessageHeaders": [* micr

## 3. `fetch` — read the user's mail

In [4]:
# fetch reads one or more entities by path via `entityUrls`.
msgs = mcp_call("fetch", {"entityUrls": ["/me/messages?$top=5&$select=subject,from,receivedDateTime"]})
for res in msgs["result"]["structuredContent"]["results"]:
    for m in res["data"].get("value", [])[:5]:
        print(m.get("receivedDateTime"), "-", m.get("subject"))

2026-07-23T14:24:46Z - Action recommended: Review Azure Copilot agent access settings before 1 August 2026
2026-07-23T05:58:35Z - Work Mate is trying to reach you in Microsoft Teams
2026-07-21T16:09:28Z - AI Instance assigned in Microsoft 365
2026-07-19T13:12:08Z - Your weekly PIM digest for CA M365 2609 (ID: a9d9510e-7131-4355-8b7e-37e7b1e99862)
2026-07-12T00:53:54Z - Your weekly PIM digest for CA M365 2609 (ID: a9d9510e-7131-4355-8b7e-37e7b1e99862)


### Recap

- One compact verb set over runtime-discovered M365 paths.
- Same security model as everything else in Work IQ — delegated, permission-aware.

**Next:** Part 4 — taking action with `do_action`.